<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>所著《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


第二章：使用预训练LLM生成文本

本笔记本中使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

&nbsp;
## 2.1 面向文本生成的 LLMs 介绍

- 本节无代码
- LLMs生成文本的方法
- 本章是准备章节：搭建我们将在全书中使用的编码环境和LLM
- 我们还将编写文本生成函数，这些函数将在后续章节中使用和扩展

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- LLM（及神经网络）流程图传统上采用自上而下的绘制与阅读方式

&nbsp;
## 2.2 设置编码环境

- 如果你正在阅读本书，很可能之前已经使用Python编写过代码
- 假如你已经搭建好Python环境（需Python 3.10或更新版本），安装依赖最简单的方式是使用`pip`：

In [2]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- 本章节的依赖项也可以手动安装：

In [3]:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

- 我推荐的方式是使用广受好评的 [uv](https://docs.astral.sh/uv/) Python包与项目管理工具
- 要安装 `uv`，请根据您的操作系统前往官方网站执行安装：https://docs.astral.sh/uv/getting-started/installation/
- 接下来，克隆GitHub仓库：

In [4]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- 如果您尚未安装 `git`，也可以从Manning网站或点击此链接手动下载源代码仓库：https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip（下载后解压）

- 在终端中，导航至 `reasoning-from-scratch` 文件夹
- 该仓库包含一个 `.python-version` 文件，确保 `uv` 默认使用与 PyTorch 兼容的 Python 版本
- 运行 `uv run jupyter lab` 启动 JupyterLab，然后打开一个空白笔记本或本章对应的笔记本
- 此命令还会自动在 `reasoning-from-scratch` 文件夹内设置本地虚拟环境（通常位于 `.venv/`），并根据 `pyproject.toml` 文件安装所有依赖项

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- 如需了解详情，请参阅 [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md) 以获取更多安装选项和细节。

&nbsp;
## 2.3 理解硬件需求与建议

- 如果你刚开始学习 PyTorch，推荐阅读我的教程《[一小时掌握 PyTorch：从张量到多 GPU 训练神经网络](https://sebastianraschka.com/teaching/pytorch-1h/)》
- 如果你已跟随前一节操作，此时应该已安装 PyTorch
- 请手动检查你的 PyTorch 安装是否支持 GPU，并查看设备支持情况：

In [5]:
import torch


print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


- 根据章节内容，代码会自动使用可用的NVIDIA (CUDA) GPU，否则在CPU上运行（若特定章节推荐，也可能使用Apple Silicon GPU）
- 第2-4章的代码在CPU上可在合理时间内执行完成
- 第5-7章的代码在CPU上执行会非常缓慢，建议这些章节使用支持CUDA的GPU（具体资源需求将在后续章节详细说明）
- 我个人更推荐使用[Lightning AI Studio](https://lightning.ai/)，该平台在注册验证后会为用户提供免费算力额度；此外，[Google Colab](https://colab.research.google.com/)也是不错的选择

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- 如需云服务器推荐，请参阅 [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md)
- 但目前阶段尚无需使用GPU；前几章内容在无GPU硬件上也能良好运行

&nbsp;
## 2.4 为 LLMs 准备输入文本

- 在本节中，我们将学习如何使用分词器；用它将输入文本转换（编码）为词元ID表示，作为大语言模型的输入。
- 我们也会使用分词器将大语言模型的输出转换（解码）回人类可读的文本表示。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- 如前所述，从头实现LLM和分词器超出了本书范围，本书专注于在已有LLM和分词器基础上从头实现推理方法
- 本书将使用预训练LLM，我们将在下一节加载该模型；此处先加载其配套分词器
- 我编写了`reasoning_from_scratch` Python包，其中包含基础LLM及对应分词器，这些代码借助[`tokenizers`](https://github.com/huggingface/tokenizers) Python库实现
- `reasoning_from_scratch`包的代码是本书配套代码的一部分，按照2.2节的说明应当已经安装

- 接下来我们下载分词器文件（这是用于 Qwen3 基础模型的分词器，更多内容将在下一节介绍）：

In [6]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

- 现在，我们可以将分词器设置从分词器文件加载到 `Qwen3Tokenizer` 中：

In [7]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 由于我们尚未加载大语言模型本身，我们将执行一个更简单的循环：将文本编码为 token IDs，然后将其解码回字符串表示形式：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [8]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [9]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [10]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


- 对于 `Qwen3Tokenizer`，大约有 15.1 万个唯一标记（词汇量）

- 关于分词的额外资源：
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) 第2章
  - [从零实现字节对编码 (BPE) 分词器](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

&nbsp;
## 2.5 加载预训练模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- 正如前一节所暗示的，本书在加载分词器时使用了 Qwen3 0.6B；经过深思熟虑选择哪个开源权重基础模型后，我最终选定了 Qwen3，因为：
  - Qwen3 是截至撰写时建模性能领先的开源权重模型
  - Qwen3 0.6B 比 Llama 3 1B 内存效率更高
  - 它既有基础模型（我们重点用其开发推理模型），也有官方推理变体可作为参考模型
- （注意："Qwen3" 的规范写法不含空格，而 "Llama 3" 包含空格）
- 秉承 "从零开始 (from-scratch)" 理念，我们使用了我用纯 PyTorch 重写的 Qwen3 实现，不依赖任何外部 LLM 库；这个从零开始的实现与原始 Qwen3 模型权重兼容
- 但本书不会详细讲解 Qwen3 的代码实现，因为那本身就需要整本书（类似我的 [《从零构建大语言模型》](https://github.com/rasbt/LLMs-from-scratch)）；相反，本书（《从零构建推理模型》）专注于在基础模型（此处为 Qwen3）之上从零开始实现推理方法
- Qwen3 模型代码详见附录 C
- 加载推理变体和更大规模 Qwen3 模型的方法详见附录 D
- 更多详情可参阅 Qwen3 [GitHub 仓库](https://github.com/QwenLM/Qwen3) 和 [技术报告](https://arxiv.org/abs/2505.09388)

- 该模型特意设计为小型化（但仍具备强大性能），以便在消费级硬件上运行
- 它可在CPU、NVIDIA GPU（`"cuda"`）、Apple Silicon GPU（`"mps"`）和Intel GPU（`"xpu"`）上流畅运行；关于性能权衡因素将在本章后续详细说明

In [11]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using Apple Silicon GPU (MPS)


- 我建议首次运行时将代码配置在 `"cpu"` 上执行，因此我们在下方硬编码了设备选择：

In [12]:
# Recommended: Use CPU on the first run-through
device = torch.device("cpu")

- 然后，我们下载包含预训练模型权重的文件，该文件大小约为1.5 GB：

In [13]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- 对于熟悉大语言模型架构的读者，我们加载的Qwen3 0.6B模型架构设计如下所示，但请注意，本书并**不**要求必须理解该架构，因为我们将在后续章节中不作修改，而是在此基础上添加推理技术

- 我从零开始编写了 [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) Python 包中的 Qwen3 模型架构；源代码也展示在附录 C 中；不过再次强调，这只是为好奇的读者提供的额外内容，阅读或理解这些内部细节并非跟读本书其余部分的必要条件。

In [14]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px" alt="第二章图8">

&nbsp;
## 2.6 理解顺序的LLM文本生成过程

- 在本节中，我们将编写一个简单的包装函数，以便使用LLM生成文本（我们将在第4章为该功能添加额外功能）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLMs 一次生成一个词：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- 上图是一个简化版本，仅显示新生成的单词；下图放大显示了第一次迭代的过程：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [15]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0))

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [16]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0))

tensor([[1, 2, 3]])
tensor([1, 2, 3])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [17]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [18]:
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3750,  2.0312,  8.0000,  ..., -2.5469, -2.5469, -2.5469],
       dtype=torch.bfloat16)


In [19]:
print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286])


In [20]:
print(tokenizer.decode([20286]))

 Large


In [21]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

tensor(3)
tensor(2)


&nbsp;
## 2.7 编写一个最小化的文本生成函数


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">  
*图片来源于 Sebastian Raschka*

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- `generate_text_basic_stream` 函数实现了此顺序文本生成过程：

In [22]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # Stop if we encounter an end-of-sequence token
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token  # Yield each token as it's generated
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

- 让我们用它来对简单的 `"Explain large language models in a single sentence."` 提示生成100个token的响应，看看它是如何工作的（我们将在后续章节中探讨推理部分）
- 以下代码运行较慢，可能需要1-3分钟才能完成，具体取决于你的电脑配置（我们将在后续章节中提升其速度）

In [23]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # Deactivates buffering so tokens are printed live
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The evolution of language has

- 请注意，LLM 能很好地遵循指令，但在 `

In [24]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


- 为方便起见，此token ID存储为tokenizer属性（eos = 序列结束）：

In [25]:
print(tokenizer.eos_token_id)

151643


- 我们可以用它告诉LLM（或者更准确地说，`generate_text_basic_stream`函数）何时停止生成文本

In [26]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id  # Use EOS token
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

- 以上是您在CPU上运行代码时得到的结果，生成的文本可能会因设备不同而略有差异。

- 在我们结束本节并探讨如何加速代码之前，让我们先实现一个简单的基准测试函数来跟踪计算性能。

In [27]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Check whether we are actually using this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Synchronize if supported (important for async backends)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [28]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 1.39 sec
29 tokens/sec


&nbsp;
## 2.8 通过KV缓存实现更快的推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- 请注意，本书中的代码注重代码可读性，关于优化的内容完全可以另写一本书
- 在此，我们探讨一种称为"KV缓存"的工程技巧（KV指代大语言模型注意力机制中的键值）
- 如果您不熟悉这些术语，请放心，您只需了解我们可以存储（缓存）在每次迭代中重复使用的中间值

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- 欲了解KV缓存工作机制的更多细节，请参阅我的文章《[从零开始理解与编写LLM中的KV缓存](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)》
- 以下是使用KV缓存的`generate_text_basic_stream`函数的修改版本

In [29]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])  # New
    model.reset_kv_cache()                           # New

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

用法与之前类似：

In [30]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 0.84 sec
49 tokens/sec


- 正如我们所见，其速度实现了数量级的提升（从每秒4个token提升至每秒28个token；测试环境为Mac Mini M4 CPU）

&nbsp;
## 2.9 通过 PyTorch 模型编译实现更快推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- 另一种能大幅提升模型推理（文本生成）速度的技术是使用 `torch.compile`
- 其用法很简单，我们只需对模型调用 `torch.compile`（如需其他选项请参阅[官方文档](https://docs.pytorch.org/docs/stable/torch.compiler_api.html)）

In [31]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # This avoids retriggering model recompilations 
    # in PyTorch 2.8 and newer
    # if the model contains code like self.pos = self.pos + 1
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# If you have issues with torch.compile on "mps" devices and get an InductorError,
# make sure you are using PyTorch 2.9 or newer

---

**Windows 注意事项 1**

- 在 Windows 上进行编译可能比较棘手
- `torch.compile()` 使用 Inductor，它会即时编译内核，并需要一个可用的 C/C++ 工具链
- 对于 CUDA，Inductor 还依赖于 Triton，可通过社区包 `triton-windows` 获取
  - 如果你看到 `cl not found`，[请安装带有“C++ 工作负载”的 Visual Studio Build Tools](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170)，并从 "x64 Native Tools" 命令提示符运行 Python
  - 如果你在使用 CUDA 时看到 `triton not found`，请安装 `triton-windows`（例如，`uv pip install "triton-windows<3.4"`）。
- 对于 CPU，一位读者进一步建议遵循这份 [Windows 版 PyTorch Inductor 指南](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)
  - 这里，重要的一点是在安装 Visual Studio 2022 时安装英文语言包，以避免 UTF-8 错误
  - 同时，请注意代码需要通过 “Visual Studio 2022 Developer Command Prompt” 运行，而不是在 notebook 中运行
- 如果此设置过程证明过于棘手，你可以跳过编译；**编译是可选的，所有代码示例不使用它也能正常运行**

**Windows 注意事项 2**

- 有读者报告称，在 Windows 上使用默认设置运行 `torch.compile` 没有加速效果；然而，使用 `"max-autotune"` 模式运行 `torch.compile` 可获得 2 倍加速：`torch.compile(model, mode="max-autotune")`

- 初始迭代可能会稍慢，因为需要完成首次编译与优化；因此我们会重复执行多次文本生成
- 首先让我们从非缓存版本开始（该过程可能较慢，大约需要xx分钟）

In [32]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

W0213 17:02:09.090000 73246 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Warm-up run


Time: 27.15 sec
1 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 1:


Time: 0.82 sec
42 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 2:


Time: 0.82 sec
42 tokens/sec

------------------------------



- 如上所述，以5 tokens/sec的速度运行，仅比之前的4 tokens/sec略快于
- 现在让我们看看KV cache版本的表现如何

In [33]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Warm-up run


Time: 45.89 sec
0 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 1:


Time: 0.48 sec
84 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 2:


Time: 0.45 sec
90 tokens/sec

------------------------------



- 如我们所见，编译带来了显著的2倍加速（64 tokens/sec 对比 30 tokens/sec）
- 以下是包含更多结果的表格

| 模型        | 模式              | 硬件                 | 每秒令牌数     | 显存 (VRAM)       |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3Model | 常规              | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | 常规（已编译）      | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | KV 缓存           | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3Model | KV 缓存（已编译）   | Mac Mini M4 CPU      | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | 常规              | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3Model | 常规（已编译）      | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3Model | KV 缓存           | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3Model | KV 缓存（已编译）   | Mac Mini M4 GPU      | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | 常规              | NVIDIA H100 GPU      | 51            | 1.55 GB           |
| Qwen3Model | 常规（已编译）      | NVIDIA H100 GPU      | 164           | 1.81 GB           |
| Qwen3Model | KV 缓存           | NVIDIA H100 GPU      | 48            | 1.52 GB           |
| Qwen3Model | KV 缓存（已编译）   | NVIDIA H100 GPU      | 141           | 1.81 GB           |
|            |                   |                      |               |                   |
| Qwen3Model | 常规              | NVIDIA DGX Spark GPU | 74            | 1.53 GB           |
| Qwen3Model | 常规（已编译）      | NVIDIA DGX Spark GPU | 103           | 1.49 GB           |
| Qwen3Model | KV 缓存           | NVIDIA DGX Spark GPU | 68            | 1.47 GB           |
| Qwen3Model | KV 缓存（已编译）   | NVIDIA DGX Spark GPU | 98            | 1.47 GB           |

- 上图所示的 NVIDIA DGX Spark 采用 GB10（Blackwell 架构）GPU
- 请注意，我们运行所有示例时均使用单条提示语（即批处理大小为1）；若对批量推理感兴趣，请参阅附录 E

&nbsp;
## 总结

- 本节无代码